# MLP vs CNN vs EfficientNetV2B0, with controlled experiments

This notebook runs the controlled experiments behind the **Model & optimizer comparison** in the [CNN Optimizer Explorer](https://febrianggi.github.io/CNN-Implementer/) dashboard, on Colab's GPU. In one place it:

1. gets the repository code and the official data,
2. shows the three models: an MLP, the small CNN, and EfficientNetV2B0 pretrained on ImageNet,
3. trains them with one protocol: the same stratified train/validation split for every model, paired seeds, and checkpoints chosen on validation macro-F1,
4. for EfficientNet, runs one warm-up (Adam, frozen backbone) and then three fine-tuning branches from that same checkpoint: Adam, AdamW and Nadam,
5. evaluates on the official test set only after training, and compares the results per seed, with curves, a confusion matrix and paired differences.

The training code is `src/experiments.py` from the repository, so the notebook and the dashboard always use the same protocol.

**Before you start:** *Runtime → Change runtime type → T4 GPU*.

<details>
<summary>Bahasa Indonesia</summary>

# MLP vs CNN vs EfficientNetV2B0, dengan eksperimen terkontrol

Notebook ini menjalankan eksperimen terkontrol di balik bagian **Perbandingan model & optimizer** pada dashboard [CNN Optimizer Explorer](https://febrianggi.github.io/CNN-Implementer/dashboard/index.id.html), memakai GPU Colab. Dalam satu tempat, notebook ini:

1. mengambil kode repositori dan data resmi,
2. menampilkan tiga model: MLP, CNN kecil, dan EfficientNetV2B0 yang sudah dilatih di ImageNet,
3. melatih ketiganya dengan satu protokol: split train/validasi terstratifikasi yang sama untuk semua model, seed berpasangan, dan checkpoint yang dipilih dengan macro-F1 validasi,
4. untuk EfficientNet, menjalankan satu warm-up (Adam, backbone dibekukan) lalu tiga cabang fine-tuning dari checkpoint yang sama: Adam, AdamW, dan Nadam,
5. mengevaluasi pada test set resmi hanya setelah training, lalu membandingkan hasilnya per seed, lengkap dengan kurva, confusion matrix, dan selisih berpasangan.

Kode training-nya adalah `src/experiments.py` dari repositori, jadi notebook dan dashboard selalu memakai protokol yang sama.

**Sebelum mulai:** *Runtime → Change runtime type → T4 GPU*.

</details>

## 1. Settings

- `DATASET`: `"cifar10"` or `"fashion"`.
- `STUDY`: `"budget"` trains on a fixed subset of 300 training and 150 validation images per class, with EfficientNet at 96 × 96. On a T4 this takes a few minutes per seed. `"full"` uses the whole training split, with EfficientNet at 224 × 224 as in the reference recipe; that is much slower (start with one seed).
- `MODELS` and `SEEDS`: which models and seeds to train. Seeds are paired: seed 1 gives every model the same shuffling and random streams.
- `QUICK_RUN = True` trains on 300 images for 2 epochs, only to check that everything works. Quick runs go to a separate folder and are never mixed with real results.

<details>
<summary>Bahasa Indonesia</summary>

## 1. Pengaturan

- `DATASET`: `"cifar10"` atau `"fashion"`.
- `STUDY`: `"budget"` melatih pada subset tetap berisi 300 citra latih dan 150 citra validasi per kelas, dengan EfficientNet pada 96 × 96. Di T4 ini butuh beberapa menit per seed. `"full"` memakai seluruh split latih, dengan EfficientNet pada 224 × 224 seperti resep referensi; ini jauh lebih lama (mulai dengan satu seed).
- `MODELS` dan `SEEDS`: model dan seed yang dilatih. Seed-nya berpasangan: seed 1 memberi setiap model urutan acak dan aliran acak yang sama.
- `QUICK_RUN = True` melatih 300 citra selama 2 epoch, hanya untuk mengecek semuanya berjalan. Run cepat disimpan di folder terpisah dan tidak pernah tercampur dengan hasil sebenarnya.

</details>

In [ ]:
DATASET = "cifar10"          # "cifar10" or "fashion"
STUDY = "budget"             # "budget" (small subset, EfficientNet 96 px) or "full" (whole split, EfficientNet 224 px)
MODELS = ["mlp", "cnn", "effnet"]
SEEDS = [1, 2, 3]
QUICK_RUN = False            # True: 300 images, 2 epochs, results in /content/smoke

REPO_URL = "https://github.com/Febrianggi/CNN-Implementer.git"
REPO_BRANCH = "main"         # the branch that contains src/experiments.py
DATA_DIR = None              # None: official files through keras.datasets; or a folder with the official files
EFFNET_WEIGHTS = "imagenet"  # or a path to an EfficientNetV2B0 no-top .h5 file

## 2. Code, GPU and libraries

The cell clones the repository (or reuses it if it is already here) and checks for a GPU. Colab already has TensorFlow and Keras 3 installed.

<details>
<summary>Bahasa Indonesia</summary>

## 2. Kode, GPU, dan library

Sel ini meng-clone repositori (atau memakai yang sudah ada) dan mengecek GPU. Colab sudah punya TensorFlow dan Keras 3.

</details>

In [ ]:
import os, sys, json, glob, subprocess
from pathlib import Path

if Path("src/experiments.py").exists():
    REPO = Path(".").resolve()
elif Path("/content/CNN-Implementer/src/experiments.py").exists():
    REPO = Path("/content/CNN-Implementer")
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, "/content/CNN-Implementer"], check=True)
    REPO = Path("/content/CNN-Implementer")
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
assert Path("src/experiments.py").exists(), "This branch has no src/experiments.py; set REPO_BRANCH."

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
import experiments as E

print("Repository:", REPO)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none (training EfficientNet will be very slow)")
OUT = Path("/content/smoke") if QUICK_RUN else E.EXP / "runs"

## 3. Data and the fixed split

The official files come through `keras.datasets`. The split is stored in `experiments/splits/<dataset>.json`: a stratified 10% of the official training set is held out for validation, and the small subset is drawn from inside that split. Every model and seed uses exactly these indices; the file also stores a fingerprint of the data, so a different copy of the dataset is refused. The official test set (10,000 images) is not touched until evaluation.

<details>
<summary>Bahasa Indonesia</summary>

## 3. Data dan split tetap

Berkas resmi diambil lewat `keras.datasets`. Split-nya tersimpan di `experiments/splits/<dataset>.json`: 10% data latih resmi yang terstratifikasi disisihkan untuk validasi, dan subset kecil diambil dari dalam split itu. Setiap model dan seed memakai indeks yang persis sama; berkas itu juga menyimpan sidik jari data, jadi salinan dataset yang berbeda akan ditolak. Test set resmi (10.000 citra) tidak disentuh sampai evaluasi.

</details>

In [ ]:
SRC = ["--source", "files", "--data-dir", str(DATA_DIR)] if DATA_DIR else []
xtr, ytr, xte, yte = E.load_official(DATASET, "files" if DATA_DIR else "keras", DATA_DIR)
split, train_idx, val_idx = E.get_split(DATASET, xtr, ytr)
tr, va = E.study_indices(STUDY, split, train_idx, val_idx)
CLASSES = E.DATASETS[DATASET]["classes"]

print(f"{E.DATASETS[DATASET]['title']}: train {len(tr)}, validation {len(va)}, test {len(yte)}")
print("Validation images per class:", np.bincount(ytr[va], minlength=10))
print("Train and validation overlap:", len(np.intersect1d(tr, va)))

plt.figure(figsize=(10, 2.2))
for i, k in enumerate(np.unique(ytr[tr], return_index=True)[1]):
    plt.subplot(1, 10, i + 1)
    img = xtr[tr[k]]
    plt.imshow(img[..., 0] if img.shape[-1] == 1 else img, cmap="gray")
    plt.title(CLASSES[ytr[tr[k]]], fontsize=8)
    plt.axis("off")
plt.show()

## 4. The three models

- **MLP**: Flatten, Dense 256, Dense 128, Dense 10. No convolution.
- **CNN**: two Conv 3 × 3 + max-pooling blocks, Dense 128 with dropout 0.3, Dense 10.
- **EfficientNetV2B0**: the images enter as pixels 0 to 255 (Fashion-MNIST's gray channel is repeated to three channels) and are resized inside the model, then flipped and rotated slightly during training. The backbone's own preprocessing rescales them and applies ImageNet's mean and standard deviation. A dropout 0.4 and a 10-class softmax replace the ImageNet head.

One detail matters for fine-tuning. The reference notebook (Keras 2.10) calls the backbone with `training=False` so its BatchNorm statistics stay fixed. Keras 3 does not pass that flag through `model.fit`, so `src/experiments.py` pins BatchNorm and stochastic depth to inference mode layer by layer; the weights themselves still train. Every run records whether the BatchNorm statistics stayed unchanged.

<details>
<summary>Bahasa Indonesia</summary>

## 4. Tiga model

- **MLP**: Flatten, Dense 256, Dense 128, Dense 10. Tanpa konvolusi.
- **CNN**: dua blok Conv 3 × 3 + max pooling, Dense 128 dengan dropout 0,3, Dense 10.
- **EfficientNetV2B0**: citra masuk sebagai piksel 0 sampai 255 (channel abu-abu Fashion-MNIST diulang menjadi tiga channel) dan di-resize di dalam model, lalu dibalik dan diputar sedikit saat training. Preprocessing bawaan backbone menskalakan ulang piksel dan menerapkan rata-rata serta simpangan baku ImageNet. Dropout 0,4 dan softmax 10 kelas menggantikan head ImageNet.

Ada satu detail penting untuk fine-tuning. Notebook referensi (Keras 2.10) memanggil backbone dengan `training=False` agar statistik BatchNorm tetap. Keras 3 tidak meneruskan flag itu lewat `model.fit`, jadi `src/experiments.py` mengunci BatchNorm dan stochastic depth ke mode inferensi per layer; bobotnya sendiri tetap dilatih. Setiap run mencatat apakah statistik BatchNorm tetap tidak berubah.

</details>

In [ ]:
shape = E.DATASETS[DATASET]["shape"]
size = E.STUDIES[STUDY]["effnet_size"]
mlp, cnn = E.build_mlp(shape), E.build_cnn(shape)
eff, base = E.build_effnet(shape, size, seed=1, weights=EFFNET_WEIGHTS)

print(f"MLP {mlp.count_params():,} parameters | CNN {cnn.count_params():,} | EfficientNetV2B0 {eff.count_params():,} "
      f"(input {size} × {size}; {sum(int(np.prod(w.shape)) for w in eff.trainable_weights):,} trained during warm-up)")

# what the backbone actually receives: the output of the augmentation layers, before its own preprocessing
probe = keras.Model(eff.input, eff.get_layer("rotate").output)
batch = probe(xtr[tr[:32]].astype("float32"), training=True).numpy()
print("Backbone input:", batch.shape, "pixel range", round(float(batch.min()), 1), "to", round(float(batch.max()), 1))
print("Recipe:", {k: E.EFFNET[k] for k in ("warm_epochs", "warm_lr", "fine_epochs", "learning_rate", "weight_decay", "dropout", "patience", "batch_size")})
del eff, base, probe

## 5. Train

Each run gets its own folder, `experiments/runs/<study>/<dataset>/<model>-<optimizer>/seed-<s>/`, and a finished run is skipped, so if Colab disconnects you can simply run this cell again. An unfinished run is kept aside as `seed-<s>.attempt-<time>` and started over.

For EfficientNet the order per seed is: warm-up (Adam 0.001, backbone frozen, 2 epochs, saved and hashed), then the Adam, AdamW and Nadam branches, each loading that warm-up checkpoint, checking its hash, and starting with a new optimizer (learning rate 6e-5, up to 23 epochs, early stopping with patience 5 on validation macro-F1).

<details>
<summary>Bahasa Indonesia</summary>

## 5. Training

Setiap run punya foldernya sendiri, `experiments/runs/<studi>/<dataset>/<model>-<optimizer>/seed-<s>/`, dan run yang sudah selesai dilewati, jadi kalau Colab terputus cukup jalankan sel ini lagi. Run yang belum selesai disimpan sebagai `seed-<s>.attempt-<waktu>` lalu diulang dari awal.

Untuk EfficientNet, urutan per seed: warm-up (Adam 0,001, backbone dibekukan, 2 epoch, disimpan dan di-hash), lalu cabang Adam, AdamW, dan Nadam, yang masing-masing memuat checkpoint warm-up itu, mengecek hash-nya, dan mulai dengan optimizer baru (learning rate 6e-5, maksimal 23 epoch, early stopping dengan patience 5 pada macro-F1 validasi).

</details>

In [ ]:
quick = ["--limit", "300", "--fine-epochs", "2", "--max-epochs", "2"] if QUICK_RUN else []
cmd = [sys.executable, "src/experiments.py", "run", "--dataset", DATASET, "--study", STUDY,
       "--models", *MODELS, "--seeds", *map(str, SEEDS), "--out", str(OUT),
       "--effnet-weights", EFFNET_WEIGHTS, *SRC, *quick]
print(" ".join(cmd))
_ = subprocess.run(cmd, check=True)

## 6. Evaluate on the test set

Only now is the official test set used: every trained run predicts all 10,000 test images from its chosen checkpoint (and the MLP and CNN also from their epoch-10 snapshot, the paired *stopped at epoch 10* baseline). Predictions, ground truth, official test IDs, the confusion matrix, accuracy and macro-F1 are saved next to the run.

<details>
<summary>Bahasa Indonesia</summary>

## 6. Evaluasi pada test set

Baru sekarang test set resmi dipakai: setiap run yang sudah dilatih memprediksi ke-10.000 citra uji dari checkpoint terpilihnya (MLP dan CNN juga dari snapshot epoch 10, pembanding berpasangan *berhenti di epoch 10*). Prediksi, ground truth, ID uji resmi, confusion matrix, akurasi, dan macro-F1 disimpan di samping run-nya.

</details>

In [ ]:
_ = subprocess.run([sys.executable, "src/experiments.py", "evaluate", "--dataset", DATASET, "--study", STUDY, "--out", str(OUT),
                "--effnet-weights", EFFNET_WEIGHTS, *SRC], check=True)

## 7. Results per seed

Accuracy and macro-F1 are recomputed here from the saved predictions, not read from a log. Δ is the paired difference from the **initial CNN** (Adam, stopped at epoch 10) with the same seed, in percentage points.

<details>
<summary>Bahasa Indonesia</summary>

## 7. Hasil per seed

Akurasi dan macro-F1 dihitung ulang di sini dari prediksi yang tersimpan, bukan dibaca dari log. Δ adalah selisih berpasangan terhadap **CNN awal** (Adam, berhenti di epoch 10) dengan seed yang sama, dalam poin persentase.

</details>

In [ ]:
rows = []
for meta in sorted((OUT / STUDY / DATASET).glob("*/seed-*/run.json")):
    info = json.loads(meta.read_text())
    if info["phase"] == "warmup" or info["status"] != "evaluated":
        continue
    hist = json.loads((meta.parent / "history.json").read_text())
    variants = [("", info["optimizer"], info["best_epoch"])]
    if info.get("snapshot_epoch"):
        variants.append((f"epoch{info['snapshot_epoch']}", info["optimizer"] + "-e10", info["snapshot_epoch"]))
    for tag, opt, best in variants:
        z = np.load(meta.parent / tag / "test_predictions.npz")
        y, p = z["y_true"].astype(int), z["y_pred"].astype(int)
        f1, _ = E.macro_f1(y, p)
        rows.append({"model": info["model"], "optimizer": opt, "seed": info["seed"], "test_acc": (p == y).mean(),
                     "macro_f1": f1, "chosen_epoch": best, "epochs": len(hist) if not tag else best,
                     "val_macro_f1": None if tag else info.get("best_val_macro_f1"), "run": str(meta.parent)})
R = pd.DataFrame(rows)
base = R[(R.model == "cnn") & (R.optimizer == "adam-e10")].set_index("seed").test_acc
R["delta_pp"] = [(a - base[s]) * 100 if s in base else np.nan for a, s in zip(R.test_acc, R.seed)]
order = ["mlp adam-e10", "mlp adam", "cnn adam-e10", "cnn adam", "effnet adam", "effnet adamw", "effnet nadam"]
R["config"] = R.model + " " + R.optimizer
summary = (R.groupby("config")
             .agg(seeds=("seed", "count"), acc_mean=("test_acc", "mean"), acc_sd=("test_acc", "std"),
                  f1_mean=("macro_f1", "mean"), f1_sd=("macro_f1", "std"), delta_pp=("delta_pp", "mean"),
                  val_f1=("val_macro_f1", "mean"))
             .reindex([c for c in order if c in set(R.config)]))
display(summary.style.format({"acc_mean": "{:.2%}", "acc_sd": "{:.2%}", "f1_mean": "{:.4f}", "f1_sd": "{:.4f}",
                              "delta_pp": "{:+.2f}", "val_f1": "{:.4f}"}, na_rep=""))
display(R.pivot_table(index="config", columns="seed", values="test_acc").reindex(summary.index).style.format("{:.2%}"))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 0.5 * len(summary) + 1))
for i, cfg in enumerate(summary.index):
    v = R[R.config == cfg].test_acc * 100
    ax.barh(i, v.mean(), color="#C9D2DF")
    ax.scatter(v, [i] * len(v), color="#2563EB", zorder=3)
ax.set_yticks(range(len(summary)), summary.index)
ax.invert_yaxis()
ax.set_xlabel("test accuracy (%)  ·  bar = mean, dots = seeds")
lo = R.test_acc.min() * 100
ax.set_xlim(max(0, lo - 5), min(100, R.test_acc.max() * 100 + 2))
plt.tight_layout(); plt.show()

## 8. Curves: did it improve, overfit or underfit?

For one seed, training and validation accuracy and loss per epoch. The dashed line is the chosen checkpoint; the grey band is EfficientNet's warm-up. Read them together: a validation loss that turns upward while training accuracy keeps rising means overfitting, curves that are both still climbing at the last epoch mean training stopped too early. A smaller train−validation gap on its own does not mean a better model; compare test accuracy too.

<details>
<summary>Bahasa Indonesia</summary>

## 8. Kurva: membaik, overfit, atau underfit?

Untuk satu seed, akurasi dan loss train serta validasi per epoch. Garis putus-putus adalah checkpoint terpilih; pita abu-abu adalah warm-up EfficientNet. Baca keduanya bersamaan: loss validasi yang berbalik naik sementara akurasi train terus naik berarti overfitting, sedangkan kedua kurva yang masih naik di epoch terakhir berarti training berhenti terlalu cepat. Selisih train−validasi yang lebih kecil saja tidak berarti model lebih baik; bandingkan juga akurasi ujinya.

</details>

In [ ]:
SEED = SEEDS[0]
runs = sorted(R[(R.seed == SEED) & ~R.optimizer.str.endswith("-e10")].itertuples(), key=lambda r: order.index(r.config))
fig, axes = plt.subplots(len(runs), 2, figsize=(11, 2.6 * len(runs)), squeeze=False)
for row, r in zip(axes, runs):
    h = pd.DataFrame(json.loads((Path(r.run) / "history.json").read_text()))
    for ax, (tr_k, va_k, name) in zip(row, [("accuracy", "val_accuracy", "accuracy"), ("loss", "val_loss", "loss")]):
        ax.plot(h.epoch, h[tr_k], label="train")
        ax.plot(h.epoch, h[va_k], "--", label="validation")
        ax.axvline(r.chosen_epoch, color="k", ls=":", lw=1)
        if r.model == "effnet":
            ax.axvspan(0.5, E.EFFNET["warm_epochs"] + 0.5, color="0.9")
        ax.set_title(f"{r.model} {r.optimizer}, seed {SEED}: {name}", fontsize=9)
        ax.xaxis.get_major_locator().set_params(integer=True)
    row[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# EfficientNet: the three fine-tuning branches of one seed, from the same warm-up checkpoint
eff_runs = sorted((OUT / STUDY / DATASET).glob(f"effnet-*/seed-{SEED}/run.json"))
infos = {json.loads(m.read_text())["optimizer"]: json.loads(m.read_text()) for m in eff_runs}
if "warmup" in infos:
    w = infos["warmup"]
    print("warm-up weights hash:", w["warmup_weights_sha256"][:16], "| backbone input range:", w.get("backbone_input_range"))
    for opt in ("adam", "adamw", "nadam"):
        if opt in infos:
            i = infos[opt]
            print(f"{opt:6s} starts from {i['start_weights_sha256'][:16]}  same as warm-up: {i['start_weights_sha256'] == w['warmup_weights_sha256']}"
                  f"  BatchNorm statistics unchanged: {i.get('batchnorm_stats_unchanged')}")
    plt.figure(figsize=(9, 3.2))
    for opt in ("adam", "adamw", "nadam"):
        p = OUT / STUDY / DATASET / f"effnet-{opt}" / f"seed-{SEED}" / "history.json"
        if p.exists():
            h = pd.DataFrame(json.loads(p.read_text()))
            plt.plot(h.epoch, h.val_accuracy, "--" if opt == "adamw" else "-", label=opt)
    plt.axvspan(0.5, E.EFFNET["warm_epochs"] + 0.5, color="0.9")
    plt.title(f"EfficientNet validation accuracy, seed {SEED}"); plt.xlabel("epoch"); plt.legend(); plt.show()
else:
    print("No EfficientNet run for this seed.")

Note on AdamW: Keras applies decoupled weight decay as `w × learning_rate × weight_decay` per step. With learning rate 6e-5 and weight decay 1e-5 that is `w × 6e-10`, below float32 precision, so the AdamW branch ends up identical to Adam. The repository's runs confirmed bit-identical weights. To see a real AdamW effect, the weight decay would need to be much larger (Keras' default is 0.004); that is a new configuration, chosen on validation data only.

<details>
<summary>Bahasa Indonesia</summary>

Catatan tentang AdamW: Keras menerapkan decoupled weight decay sebagai `w × learning_rate × weight_decay` per langkah. Dengan learning rate 6e-5 dan weight decay 1e-5, nilainya `w × 6e-10`, lebih kecil dari presisi float32, sehingga cabang AdamW berakhir identik dengan Adam. Run di repositori memastikan bobotnya identik bit demi bit. Untuk melihat efek AdamW yang nyata, weight decay perlu jauh lebih besar (bawaan Keras 0,004); itu konfigurasi baru, yang dipilih hanya dengan data validasi.

</details>

## 9. Where the best model goes wrong

The confusion matrix of the configuration with the highest **validation** macro-F1 (the test set is not used to pick it), for the first seed.

<details>
<summary>Bahasa Indonesia</summary>

## 9. Di mana model terbaik salah

Confusion matrix konfigurasi dengan macro-F1 **validasi** tertinggi (test set tidak dipakai untuk memilihnya), untuk seed pertama.

</details>

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
cand = R[R.val_macro_f1.notna()].groupby("config").val_macro_f1.mean()
best_cfg = cand.idxmax()
r = R[(R.config == best_cfg) & (R.seed == R[R.config == best_cfg].seed.min())].iloc[0]
m = json.loads((Path(r.run) / "metrics.json").read_text())
print(f"Chosen on validation: {best_cfg} (val macro-F1 {cand.max():.4f}); seed {r.seed}: test accuracy {m['test_accuracy']:.2%}, macro-F1 {m['test_macro_f1']:.4f}")
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(np.array(m["confusion"]), display_labels=CLASSES).plot(ax=ax, cmap="Blues", xticks_rotation=60, colorbar=False)
plt.tight_layout(); plt.show()

## 10. Keep the results

The run folders hold everything the dashboard needs except the checkpoints. To bring Colab runs into the repository: download the zip below, unpack it into `experiments/runs/` of your local copy, then run

```bash
python src/export_experiments.py --jpeg-dir data/CIFAR-10-images
python src/check_experiments.py
```

and commit the changed files. The dashboard will show the new runs and drop their *not trained yet* status.

<details>
<summary>Bahasa Indonesia</summary>

## 10. Menyimpan hasil

Folder run berisi semua yang dibutuhkan dashboard kecuali checkpoint. Untuk membawa run dari Colab ke repositori: unduh zip di bawah, ekstrak ke `experiments/runs/` di salinan lokalmu, lalu jalankan

```bash
python src/export_experiments.py --jpeg-dir data/CIFAR-10-images
python src/check_experiments.py
```

lalu commit berkas yang berubah. Dashboard akan menampilkan run baru dan status *belum dilatih*-nya hilang.

</details>

In [ ]:
import shutil, zipfile
zip_path = Path(f"/content/runs_{STUDY}_{DATASET}{'_quick' if QUICK_RUN else ''}.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in (OUT / STUDY / DATASET).rglob("*"):
        if f.is_file() and not f.name.endswith(".weights.h5") and ".attempt-" not in str(f):
            z.write(f, f.relative_to(OUT))
print(zip_path, f"{zip_path.stat().st_size / 1e6:.1f} MB")
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    pass